In [ ]:
from arize import ArizeClient
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
DATASET_ID = "RGF0YXNldDozNDE5OTU6dGZ4YQ==" # taken from 02_dataset.ipynb

import os, certifi
CA = certifi.where()

os.environ["SSL_CERT_FILE"] = CA
os.environ["REQUESTS_CA_BUNDLE"] = CA
os.environ["CURL_CA_BUNDLE"] = CA

print("Using CA:", CA)


In [ ]:
client = ArizeClient(api_key=API_KEY)

In [ ]:
ds = client.datasets.list_examples(dataset=DATASET_ID, space=SPACE_ID)
examples = ds.to_dict()["examples"]
examples

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4.1-2025-04-14", temperature=0.0)
# chain = llm | StrOutputParser()

def sample_task(dataset_row) -> str:
  return dataset_row.get("question")
  # return chain.invoke(dataset_row.get("question"))

In [ ]:
sample_task(examples[0])

In [ ]:
from typing import Any, Optional, Mapping
from arize.experiments import EvaluationResult

def is_correct(output, dataset_row):
    expected = dataset_row.get("question")
    correct = expected in output
    return EvaluationResult(
        score=int(correct),
        label="correct" if correct else "incorrect",
        explanation="Evaluator explanation here"
    )

In [ ]:
experiment, experiment_df = client.experiments.run(
    name="basic-experiment-2",
    dataset=DATASET_ID,
    task=sample_task,
    evaluators=[is_correct],
    concurrency=10,
    exit_on_error=False,
    dry_run=False,
)